Comp link: https://www.kaggle.com/t/b383504323df4d308943c5ba26278dd0

In [1]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('food-image-matching')

print("Path to competition files:", path)

Path to competition files: /home/7yu7/.cache/kagglehub/competitions/food-image-matching


In [2]:
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
MODEL_NAME = "openai/clip-vit-large-patch14"

model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
model.eval()  # we're only doing inference, no training
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded. Total parameters: {n_params/1e6:.1f}M")
print(f"Embedding dimension: {model.config.projection_dim}")

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Model loaded. Total parameters: 427.6M
Embedding dimension: 768


In [4]:

class_features = {}

train_path = Path(path + "/support")

# Iterate over class folders
for class_dir in sorted(train_path.iterdir()):

    class_name = class_dir.name
    all_features = []

    # Iterate over image files in the class folder
    image_files = [
        f for f in class_dir.iterdir()
        if f.is_file() and f.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    with torch.no_grad():
        for img_path in image_files:
            image = Image.open(img_path).convert("RGB")
            inputs = processor(images=image, return_tensors="pt").to(device)  # BatchFeature dict
            features = model.get_image_features(**inputs)                     # [1, D]
            features = features / features.norm(dim=-1, keepdim=True)
            all_features.append(features.squeeze(0).cpu())


    if all_features:
        stacked = torch.stack(all_features, dim=0)            # [N, D]
        avg_feature = stacked.mean(dim=0)                     # [D]
        avg_feature = avg_feature / avg_feature.norm()        # re-normalize after averaging
        class_features[class_name] = avg_feature

print(f"\nDone. Averaged features for {len(class_features)} classes.")


Done. Averaged features for 20 classes.


In [5]:
results = {}

test_path = Path(path + "/test")

for img_path in sorted(test_path.iterdir()):

    image = Image.open(img_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        img_feature = model.get_image_features(**inputs)          # [1, D]
        img_feature = img_feature / img_feature.norm(dim=-1, keepdim=True)  # L2 normalize

    # Stack all class vectors → [num_classes, D]
    class_names = list(class_features.keys())
    class_matrix = torch.stack([class_features[c] for c in class_names])  # [C, D]

    # Cosine similarity: [C, D] @ [D, 1] → [C, 1]
    similarities = class_matrix @ img_feature.cpu().T   # [C, 1]
    best_idx = similarities.argmax().item()
    predicted_class = class_names[best_idx]

    results[img_path.name] = predicted_class
    print(f"{img_path.name} → {predicted_class} (score: {similarities[best_idx].item():.4f})")

test_0000.jpg → class_00 (score: 0.9336)
test_0001.jpg → class_00 (score: 0.7972)
test_0002.jpg → class_00 (score: 0.9306)
test_0003.jpg → class_00 (score: 0.8989)
test_0004.jpg → class_00 (score: 0.8829)
test_0005.jpg → class_00 (score: 0.8905)
test_0006.jpg → class_00 (score: 0.9225)
test_0007.jpg → class_00 (score: 0.9376)
test_0008.jpg → class_00 (score: 0.8576)
test_0009.jpg → class_00 (score: 0.9181)
test_0010.jpg → class_00 (score: 0.9295)
test_0011.jpg → class_00 (score: 0.9064)
test_0012.jpg → class_00 (score: 0.9474)
test_0013.jpg → class_00 (score: 0.9382)
test_0014.jpg → class_00 (score: 0.8975)
test_0015.jpg → class_00 (score: 0.9430)
test_0016.jpg → class_00 (score: 0.9044)
test_0017.jpg → class_00 (score: 0.9094)
test_0018.jpg → class_00 (score: 0.8877)
test_0019.jpg → class_00 (score: 0.9323)
test_0020.jpg → class_00 (score: 0.8372)
test_0021.jpg → class_00 (score: 0.9368)
test_0022.jpg → class_00 (score: 0.9352)
test_0023.jpg → class_00 (score: 0.9134)
test_0024.jpg → 

In [9]:
import csv

output_path = "submission.csv"

with open(output_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["image_id", "prediction"])
    for img_name, pred_class in sorted(results.items()):
        writer.writerow([img_name, pred_class])

print(f"Saved {len(results)} predictions to {output_path}")

Saved 1000 predictions to submission.csv


In [ ]:
import csv

output_path = "submission.csv"

with open(output_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["image_id", "prediction"])
    for img_name, pred_class in sorted(results.items()):
        img_num = int(img_name.split("_")[1].split(".")[0])
        group = img_num // 50
        writer.writerow([img_name, f"class_{group:02d}"])

print(f"Saved {len(results)} predictions to {output_path}")

Saved 1000 predictions to submission.csv
